# DorAImon — Fine-tune Ministral-3B for Screen Intent Classification

Fine-tunes `mistralai/Ministral-3-3B-Instruct-2512` with 4-bit QLoRA to classify screen OCR JSON into one of 4 intents:
- `hesitation_coding` — user stalled on code
- `foreign_language` — non-English text on screen
- `error_visible` — error/exception in terminal or editor
- `typing_fluent` — normal activity, no intervention needed

Output is a GGUF model ready for Ollama local deployment in the DorAImon Electron overlay.

**Runtime**: T4 GPU, ~20 min training

In [ ]:
# ============================================================
# Cell 1: Environment Setup + W&B Init
# ============================================================

!pip install -q \
    torch \
    transformers>=4.44.0 \
    peft>=0.12.0 \
    trl>=0.9.0 \
    bitsandbytes>=0.43.0 \
    datasets \
    accelerate>=0.33.0 \
    wandb \
    scikit-learn

import os
import json
import random
import numpy as np
import torch
import wandb
from google.colab import userdata

# Seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── W&B Init (exact entity/project pattern) ──
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

run = wandb.init(
    entity="42tokyo",
    project="doraimon-ministral-finetune",
    config={
        "model": "Ministral-3B",
        "model_id": "mistralai/Ministral-3-3B-Instruct-2512",
        "task": "intent_classification",
        "intents": ["hesitation", "foreign_language", "error_visible", "typing_fluent"],
        "lora_r": 16,
        "lora_alpha": 32,
        "epochs": 3,
        "lr": 2e-4,
        "batch_size": 1,
        "gradient_accumulation_steps": 8,
        "quantization": "4bit-nf4",
        "dataset_size": 100,
    },
)

print(f"W&B run: {run.url}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================
# Cell 2: Synthetic Dataset — 100 OCR JSON → Intent JSON pairs
# ============================================================

from datasets import Dataset

def make_hesitation_examples(n=25):
    """Code editor with cursor stalled >3s."""
    examples = []
    editors = ["vscode", "vim", "neovim", "jetbrains", "sublime"]
    files = ["main.py", "app.js", "index.tsx", "server.go", "lib.rs", "utils.ts", "handler.py"]
    for i in range(n):
        stall = round(random.uniform(3.1, 45.0), 1)
        lines = random.randint(5, 500)
        cursor_line = random.randint(1, lines)
        editor = random.choice(editors)
        fname = random.choice(files)
        elements = [
            {"type": "code_editor", "editor": editor, "file": fname,
             "cursor_line": cursor_line, "total_lines": lines,
             "cursor_stalled_seconds": stall},
        ]
        # Sometimes add a terminal that's also idle
        if random.random() > 0.5:
            elements.append({"type": "terminal", "last_command_age_seconds": round(stall + random.uniform(0, 10), 1)})
        # Sometimes add browser with docs open (researching)
        if random.random() > 0.7:
            elements.append({"type": "browser", "tab_title": random.choice([
                "Stack Overflow - How to", "MDN Web Docs", "Python docs", "Rust book"
            ])})

        inp = {"screen_elements": elements}
        out = {
            "intent": "hesitation_coding",
            "confidence": round(min(1.0, 0.7 + stall / 100), 2),
            "reason": f"cursor stalled {stall}s in {editor} at {fname}:{cursor_line}"
        }
        examples.append((inp, out))
    return examples


def make_foreign_language_examples(n=25):
    """Screen with non-English text blocks."""
    examples = []
    lang_samples = {
        "ja": [
            "\u30a8\u30e9\u30fc\u304c\u767a\u751f\u3057\u307e\u3057\u305f",
            "\u30d5\u30a1\u30a4\u30eb\u304c\u898b\u3064\u304b\u308a\u307e\u305b\u3093",
            "\u63a5\u7d9a\u304c\u30bf\u30a4\u30e0\u30a2\u30a6\u30c8\u3057\u307e\u3057\u305f",
            "\u30c7\u30fc\u30bf\u30d9\u30fc\u30b9\u306b\u63a5\u7d9a\u3067\u304d\u307e\u305b\u3093",
            "\u8a2d\u5b9a\u3092\u4fdd\u5b58\u3057\u307e\u3057\u305f",
        ],
        "zh": [
            "\u65e0\u6cd5\u8fde\u63a5\u670d\u52a1\u5668",
            "\u8bf7\u8f93\u5165\u5bc6\u7801",
            "\u6587\u4ef6\u4e0d\u5b58\u5728",
            "\u64cd\u4f5c\u5df2\u5b8c\u6210",
        ],
        "ko": [
            "\uc624\ub958\uac00 \ubc1c\uc0dd\ud588\uc2b5\ub2c8\ub2e4",
            "\ud30c\uc77c\uc744 \ucc3e\uc744 \uc218 \uc5c6\uc2b5\ub2c8\ub2e4",
            "\uc811\uc18d\uc774 \uac70\ubd80\ub418\uc5c8\uc2b5\ub2c8\ub2e4",
        ],
        "ar": [
            "\u062e\u0637\u0623 \u0641\u064a \u0627\u0644\u0627\u062a\u0635\u0627\u0644",
            "\u0627\u0644\u0645\u0644\u0641 \u063a\u064a\u0631 \u0645\u0648\u062c\u0648\u062f",
        ],
    }
    for i in range(n):
        lang = random.choice(list(lang_samples.keys()))
        text = random.choice(lang_samples[lang])
        ratio = round(random.uniform(0.15, 0.95), 2)
        elements = [
            {"type": "text_block", "text": text, "language": lang,
             "foreign_char_ratio": ratio},
        ]
        if random.random() > 0.4:
            elements.append({"type": "browser", "tab_title": random.choice([
                "Yahoo! JAPAN", "Naver", "Baidu", "\u30cb\u30b3\u30cb\u30b3\u52d5\u753b",
            ])})
        if random.random() > 0.6:
            elements.append({"type": "code_editor", "file": "i18n.json", "cursor_stalled_seconds": 0.5})

        inp = {"screen_elements": elements}
        out = {
            "intent": "foreign_language",
            "confidence": round(min(1.0, 0.6 + ratio * 0.4), 2),
            "reason": f"{lang} text detected ({ratio:.0%} foreign chars)"
        }
        examples.append((inp, out))
    return examples


def make_error_examples(n=25):
    """Terminal/console with error messages."""
    examples = []
    error_messages = [
        "TypeError: Cannot read properties of undefined (reading 'map')",
        "SyntaxError: Unexpected token '<'",
        "ReferenceError: useState is not defined",
        "ENOENT: no such file or directory, open '/app/config.json'",
        "ECONNREFUSED 127.0.0.1:5432",
        "ModuleNotFoundError: No module named 'torch'",
        "Traceback (most recent call last):\n  File 'main.py', line 42",
        "error[E0308]: mismatched types\n  expected `&str`, found `String`",
        "panic: runtime error: index out of range [5] with length 3",
        "java.lang.NullPointerException at com.app.Main.run(Main.java:88)",
        "Segmentation fault (core dumped)",
        "OSError: [Errno 28] No space left on device",
        "ConnectionRefusedError: [Errno 111] Connection refused",
        "KeyError: 'user_id'",
        "IndexError: list index out of range",
    ]
    for i in range(n):
        err = random.choice(error_messages)
        elements = [
            {"type": "console_output", "text": err, "has_error": True,
             "error_count": random.randint(1, 5)},
        ]
        if random.random() > 0.3:
            elements.append({"type": "code_editor", "file": random.choice([
                "main.py", "app.js", "index.tsx", "server.go"
            ]), "cursor_stalled_seconds": round(random.uniform(0.2, 2.0), 1)})
        if random.random() > 0.7:
            elements.append({"type": "browser", "tab_title": "Stack Overflow - " + err.split(":")[0]})

        inp = {"screen_elements": elements}
        out = {
            "intent": "error_visible",
            "confidence": round(random.uniform(0.85, 0.99), 2),
            "reason": f"error detected: {err[:60]}"
        }
        examples.append((inp, out))
    return examples


def make_fluent_examples(n=25):
    """Normal coding activity, no intervention needed."""
    examples = []
    for i in range(n):
        stall = round(random.uniform(0.1, 2.9), 1)
        elements = [
            {"type": "code_editor", "editor": random.choice(["vscode", "vim", "neovim"]),
             "file": random.choice(["main.py", "app.js", "index.tsx", "utils.ts"]),
             "cursor_line": random.randint(1, 300),
             "total_lines": random.randint(50, 500),
             "cursor_stalled_seconds": stall},
        ]
        if random.random() > 0.5:
            elements.append({"type": "terminal", "last_command_age_seconds": round(random.uniform(0.1, 5.0), 1)})
        if random.random() > 0.6:
            elements.append({"type": "browser", "tab_title": random.choice([
                "localhost:3000", "GitHub - Pull Request", "Figma", "Notion"
            ])})

        inp = {"screen_elements": elements}
        out = {
            "intent": "typing_fluent",
            "confidence": round(random.uniform(0.6, 0.85), 2),
            "reason": "user actively coding, no assistance needed"
        }
        examples.append((inp, out))
    return examples


# ── Generate all examples ──
all_examples = (
    make_hesitation_examples(25) +
    make_foreign_language_examples(25) +
    make_error_examples(25) +
    make_fluent_examples(25)
)
random.shuffle(all_examples)

# ── Format for Mistral instruct ──
def format_example(inp, out):
    return f"<s>[INST] Classify the user intent from this screen state:\n{json.dumps(inp)} [/INST]{json.dumps(out)}</s>"

formatted = [{"text": format_example(inp, out)} for inp, out in all_examples]

# ── 80/20 split ──
split_idx = int(len(formatted) * 0.8)
train_data = formatted[:split_idx]
val_data = formatted[split_idx:]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

# ── Log dataset to W&B ──
wandb.log({"dataset_size": len(formatted), "train_size": len(train_data), "val_size": len(val_data)})

intent_counts = {}
for _, out in all_examples:
    intent_counts[out["intent"]] = intent_counts.get(out["intent"], 0) + 1
wandb.log({"intent_distribution": wandb.plot.bar(
    wandb.Table(data=[[k, v] for k, v in intent_counts.items()], columns=["intent", "count"]),
    "intent", "count", title="Dataset Intent Distribution"
)})

print(f"Dataset: {len(train_data)} train / {len(val_data)} val")
print(f"Intent counts: {intent_counts}")
print(f"\nSample:\n{formatted[0]['text'][:300]}...")

In [ ]:
# ============================================================
# Cell 3: Load Ministral-3B with 4-bit QLoRA
# ============================================================

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_NAME = "mistralai/Ministral-3-3B-Instruct-2512"

# ── 4-bit quantization config ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ── Load model ──
print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    token=os.environ.get("HF_TOKEN"),
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ.get("HF_TOKEN"))
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── Prepare for k-bit training ──
model = prepare_model_for_kbit_training(model)

# ── LoRA config ──
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

# ── Log model info ──
trainable, total = model.get_nb_trainable_parameters()
wandb.config.update({
    "trainable_params": trainable,
    "total_params": total,
    "trainable_pct": f"{100 * trainable / total:.2f}%",
})

print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB allocated")

In [ ]:
# ============================================================
# Cell 4: Training Arguments
# ============================================================

from transformers import TrainingArguments

OUTPUT_DIR = "./ministral-intent-finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="wandb",
    run_name="ministral-3b-intent-qlora",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    seed=SEED,
    dataloader_pin_memory=False,
)

print("Training args configured:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  LR: {training_args.learning_rate}")
print(f"  Eval every {training_args.eval_steps} steps")
print(f"  Save every {training_args.save_steps} steps")

In [ ]:
# ============================================================
# Cell 5: Train
# ============================================================

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    max_seq_length=512,
    dataset_text_field="text",
    packing=False,
)

print("Starting training...")
print(f"  Train examples: {len(train_dataset)}")
print(f"  Val examples:   {len(val_dataset)}")
print(f"  Steps/epoch:    ~{len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print()

train_result = trainer.train()

# ── Log final training metrics ──
wandb.log({
    "final_train_loss": train_result.metrics["train_loss"],
    "train_runtime_seconds": train_result.metrics["train_runtime"],
    "train_samples_per_second": train_result.metrics["train_samples_per_second"],
})

print(f"\nTraining complete!")
print(f"  Loss: {train_result.metrics['train_loss']:.4f}")
print(f"  Runtime: {train_result.metrics['train_runtime']:.0f}s")

In [ ]:
# ============================================================
# Cell 6: Evaluation — JSON accuracy + confusion matrix
# ============================================================

from sklearn.metrics import confusion_matrix

INTENTS = ["hesitation_coding", "foreign_language", "error_visible", "typing_fluent"]

def predict_intent(screen_json):
    """Run inference on a single screen state JSON."""
    prompt = f"<s>[INST] Classify the user intent from this screen state:\n{json.dumps(screen_json)} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=0.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract the part after [/INST]
    if "[/INST]" in decoded:
        response = decoded.split("[/INST]")[-1].replace("</s>", "").strip()
    else:
        response = decoded[len(prompt):].strip()

    # Try to parse JSON
    try:
        parsed = json.loads(response)
        return parsed, True
    except json.JSONDecodeError:
        # Try to extract JSON from messy output
        import re
        match = re.search(r'\{[^}]+\}', response)
        if match:
            try:
                parsed = json.loads(match.group())
                return parsed, True
            except json.JSONDecodeError:
                pass
        return {"raw": response}, False


# ── Evaluate on held-out validation set ──
print("Evaluating on validation set...\n")

val_inputs = [inp for inp, _ in all_examples[split_idx:]]
val_labels = [out["intent"] for _, out in all_examples[split_idx:]]

predictions = []
true_labels = []
json_parse_successes = 0
prediction_rows = []

for i, (screen_json, true_intent) in enumerate(zip(val_inputs, val_labels)):
    pred, parsed_ok = predict_intent(screen_json)
    pred_intent = pred.get("intent", "unknown")
    pred_confidence = pred.get("confidence", 0.0)

    if parsed_ok:
        json_parse_successes += 1

    predictions.append(pred_intent)
    true_labels.append(true_intent)

    correct = pred_intent == true_intent
    status = "correct" if correct else "WRONG"
    print(f"  [{i+1:2d}/{len(val_inputs)}] {status:7s}  true={true_intent:20s}  pred={pred_intent:20s}  conf={pred_confidence}")

    prediction_rows.append([
        i + 1,
        true_intent,
        pred_intent,
        pred_confidence,
        parsed_ok,
        correct,
        json.dumps(screen_json)[:200],
    ])

# ── Calculate metrics ──
intent_accuracy = sum(1 for p, t in zip(predictions, true_labels) if p == t) / len(true_labels)
json_accuracy = json_parse_successes / len(val_inputs)

print(f"\n{'='*50}")
print(f"JSON parse success: {json_parse_successes}/{len(val_inputs)} ({json_accuracy:.0%})")
print(f"Intent accuracy:    {sum(1 for p, t in zip(predictions, true_labels) if p == t)}/{len(true_labels)} ({intent_accuracy:.0%})")

# ── Log metrics to W&B ──
wandb.log({
    "json_accuracy": json_accuracy,
    "intent_accuracy": intent_accuracy,
    "json_parse_failures": len(val_inputs) - json_parse_successes,
})

# ── Confusion matrix ──
# Map predictions to known intents for the matrix
mapped_preds = [p if p in INTENTS else "unknown" for p in predictions]
labels_for_matrix = INTENTS + (["unknown"] if "unknown" in mapped_preds else [])

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=true_labels,
        preds=mapped_preds,
        class_names=labels_for_matrix,
        title="Intent Classification Confusion Matrix",
    )
})

# ── Predictions table ──
wandb.log({
    "predictions": wandb.Table(
        columns=["#", "true_intent", "pred_intent", "confidence", "json_ok", "correct", "input_preview"],
        data=prediction_rows,
    )
})

# ── Per-intent accuracy ──
for intent in INTENTS:
    mask = [t == intent for t in true_labels]
    if any(mask):
        intent_preds = [p for p, m in zip(predictions, mask) if m]
        acc = sum(1 for p in intent_preds if p == intent) / len(intent_preds)
        wandb.log({f"accuracy_{intent}": acc})
        print(f"  {intent:20s}: {acc:.0%}")

In [ ]:
# ============================================================
# Cell 7: Save Model + W&B Artifact
# ============================================================

# ── Save locally ──
print("Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ── Merge LoRA weights for export ──
print("Merging LoRA weights...")
merged_model = model.merge_and_unload()

MERGED_DIR = "./ministral-intent-merged"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# ── Upload to W&B as artifact ──
print("Uploading to W&B...")
artifact = wandb.Artifact(
    name="ministral-3b-intent-qlora",
    type="model",
    description="Ministral-3B fine-tuned for DorAImon screen intent classification (4 intents)",
    metadata={
        "base_model": MODEL_NAME,
        "intents": INTENTS,
        "json_accuracy": json_accuracy,
        "intent_accuracy": intent_accuracy,
    },
)
artifact.add_dir(OUTPUT_DIR)
wandb.log_artifact(artifact)

# ── Final summary ──
wandb.run.summary["final_json_accuracy"] = json_accuracy
wandb.run.summary["final_intent_accuracy"] = intent_accuracy
wandb.run.summary["model_saved"] = True

wandb.finish()

print(f"\nDone!")
print(f"  LoRA adapter: {OUTPUT_DIR}/")
print(f"  Merged model: {MERGED_DIR}/")
print(f"  W&B artifact: ministral-3b-intent-qlora")

# Show saved files
import subprocess
subprocess.run(["ls", "-lh", MERGED_DIR])

In [ ]:
# ============================================================
# Cell 8: Local Deployment — GGUF Conversion + Ollama Setup
# ============================================================

# ── Step 1: Convert HF → GGUF ──
print("="*60)
print("STEP 1: Convert to GGUF")
print("="*60)
print("""
# Clone llama.cpp (if not already)
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp && make -j

# Convert merged HF model to GGUF (F16)
python convert_hf_to_gguf.py ../ministral-intent-merged \\
    --outfile ministral-intent-f16.gguf \\
    --outtype f16

# Quantize to Q4_K_M (best quality/size tradeoff, ~1.8GB)
./llama-quantize ministral-intent-f16.gguf \\
    ministral-intent-q4km.gguf Q4_K_M
""")

# ── Step 2: Ollama Modelfile ──
print("="*60)
print("STEP 2: Ollama Modelfile")
print("="*60)

modelfile_content = """FROM ./ministral-intent-q4km.gguf

TEMPLATE """{{- if .System }}<s>[INST] {{ .System }}
{{ end }}{{ .Prompt }} [/INST]{{ .Response }}</s>"""

SYSTEM """You are DorAImon's intent classifier. Given a JSON object describing screen elements (code editors, terminals, browsers, text blocks), classify the user's current intent.

Output ONLY a JSON object with these fields:
- intent: one of ["hesitation_coding", "foreign_language", "error_visible", "typing_fluent"]
- confidence: float 0.0-1.0
- reason: brief explanation

Rules:
- hesitation_coding: cursor stalled >3s, user appears stuck
- foreign_language: non-English text detected (CJK, Arabic, Cyrillic, etc.)
- error_visible: error messages, exceptions, or tracebacks in console/terminal
- typing_fluent: normal activity, no intervention needed"""

PARAMETER temperature 0.05
PARAMETER top_p 0.9
PARAMETER num_predict 128
PARAMETER stop "</s>"
PARAMETER num_ctx 512
"""

# Write Modelfile
modelfile_path = "./Modelfile"
with open(modelfile_path, "w") as f:
    f.write(modelfile_content)

print(f"Modelfile written to {modelfile_path}")
print()
print(modelfile_content)

# ── Step 3: Create & Test ──
print("="*60)
print("STEP 3: Create Ollama Model & Test")
print("="*60)
print("""
# Create the custom model in Ollama
ollama create ministral-intent-custom -f Modelfile

# Test it
ollama run ministral-intent-custom 'Classify the user intent from this screen state:
{"screen_elements": [{"type": "console_output", "text": "TypeError: Cannot read properties of undefined", "has_error": true}]}'

# Expected output:
# {"intent": "error_visible", "confidence": 0.95, "reason": "TypeError detected in console"}
""")

# ── Step 4: DorAImon .env config ──
print("="*60)
print("STEP 4: DorAImon .env Configuration")
print("="*60)
print("""
# Add to your DorAImon .env file:
USE_LOCAL_VLM=true
OLLAMA_ENDPOINT=http://localhost:11434
OLLAMA_MODEL=ministral-intent-custom

# The pipeline will now use your fine-tuned model:
#   Screenshot → Tesseract OCR → ministral-intent-custom (local) → Router → Agent
#   Cost: $0.00 per inference
#   Latency: ~100-300ms on M-series Mac
""")

print("\nNotebook complete! Next steps:")
print("  1. Download ministral-intent-merged/ from Colab")
print("  2. Convert to GGUF with llama.cpp (commands above)")
print("  3. Create Ollama model with the Modelfile")
print("  4. Set USE_LOCAL_VLM=true in DorAImon .env")
print("  5. Run DorAImon — free local intent classification!")